With random restarts, the same as ../3_1_pymc.ipynb

In [ ]:
import numpy as np
import pymc as pm
import pytensor
pytensor.config.cxx = '/usr/bin/clang++'

In [ ]:
mu_prior, sigma_prior = 0, 10
sigma_like = 1.0
max_iters = 300_000

def run_restart(seed):
    with pm.Model() as single_model:
        # Define priors
        mu = pm.Normal("mu", mu_prior, sigma_prior)
        likelihood = pm.Normal('y', mu=mu, sigma=sigma_like, observed=np.array([]))
        advi = pm.ADVI(random_seed=seed)
        single_tracker = pm.callbacks.Tracker(
                mean = advi.approx.mean.eval,
                std = advi.approx.std.eval
            )
        single_fit = advi.fit(
            max_iters, callbacks=[single_tracker])

    with pm.Model() as multi_model:
        # Define priors
        mu = pm.Normal("mu", mu_prior, sigma_prior)
        likelihood = pm.Normal('y', mu=mu, sigma=sigma_like, observed=np.array([]))
        advi_2 = pm.ADVI(random_seed=seed + 1000)
        multi_tracker = pm.callbacks.Tracker(
                mean = advi_2.approx.mean.eval,
                std = advi_2.approx.std.eval
            )
        multi_fit = advi_2.fit(
            max_iters, callbacks=[multi_tracker],
            obj_n_mc=100)
    return single_tracker, multi_tracker

In [ ]:
# using gpu, run 100 restarts in parallel and plot some
# informative summaries of the 100 restarts, e.g.
# the distribution of the final mean and std 
# estimates across the 100 restarts and a plot 
# of the mean and std estimates across iterations
# for a few random restarts to show the variability 
# in convergence behavior across restarts
from tqdm import tqdm
seeds = np.arange(100)
# results = [run_restart(seed) for seed in tqdm(seeds)]

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import os
import numpy as np

seeds = np.arange(100)

results = [None] * len(seeds)

with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
    futures = {ex.submit(run_restart, int(seed)): int(seed) for seed in seeds}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        seed = futures[fut]
        results[seed] = fut.result()


Output()